# s12 — Durable Task Graph

**What this teaches:** how to give the agent a **persistent, dependency-aware task list**. Where [s11_todo](../s11_todo/s11_todo.ipynb) is a scratch list, the task graph from `internal/tasks` survives across runs and supports `depends_on` so steps can be ordered or blocked.

**Concept anchor:** the leader records intent as a graph (`task_create` / `task_update` / `task_list`). The JSON file persists at `logs/agent_tasks_<u>_<ts>.json`. Re-running the agent picks up where it left off, and dependencies prevent a step from being marked `in_progress` while its parent is still pending.

## Prerequisites

- LLM provider configured. Default is `openai_compat` for self-hosted Ollama / vLLM — see [docs/providers.md](../../docs/providers.md).
- Working directory should allow writes under `logs/` (the default location for the task JSON).

## 1. Imports

`tasks` exposes `New` (back-compat single-session constructor) and `NewSessionScoped` (for concurrent sessions). For learning purposes we use the simpler `New`.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    "github.com/blouargant/yoke/internal/tasks"
)

## 2. Helper

Panic-based `must` so a notebook error doesn't kill the kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the model + task graph

`tasks.New("")` creates a graph backed by an auto-named JSON file under `logs/`. Pass a path string here to control where it lands.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

g := tasks.New("")
fmt.Println("task graph ready")

## 4. Build the agent with task tools

`g.Tools()` returns the three task-graph tools (`task_create`, `task_update`, `task_list`). The `Instruction` nudges the model to record intent **before** executing — without that, the model often forgets to plan.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s12_tasks",
    Description: "Durable task-graph demo.",
    Instruction: "When asked to plan, use task_create to record each step (with depends_on if needed) before executing.",
    Model:       llm,
    Tools:       g.Tools(),
})
must(err)
r, err := agentkit.Runner("s12", a)
must(err)

## 5. Run

The prompt deliberately specifies a chain of dependencies (3 depends on 2; 2 depends on 1). Watch the model call `task_create` three times with `depends_on` set correctly, then `task_list` to confirm the graph.

In [ ]:
prompt := "Plan three tasks for shipping a Go CLI: 1) write README, 2) add tests (depends on README), 3) tag v1.0 (depends on tests). Then list them."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- A JSON file appeared under `logs/agent_tasks_*.json`. Open it — that's the durable state. If you re-run the agent with the same session ID, this state is loaded.
- Compare to [s11_todo](../s11_todo/s11_todo.ipynb): todo is *plain* and ephemeral; tasks are *graph-shaped* and durable.
- The `task_update` tool gates each transition — a task can't be `in_progress` while its parent is `pending`.

## Try it yourself

1. Re-run the notebook with a follow-up prompt like "mark task 1 as completed and start task 2" — the graph state from the JSON file is reused.
2. Edit the JSON file by hand, then ask the agent to list tasks — your changes are honored.